# SGX Multi-Factor FYP — `backtest_survivorship.ipynb`
### Survivorship-bias correction via point-in-time index membership

The base pipeline ranks **today's** 30 STI/SGMCNL constituents backfilled to 2001 — a
survivorship-biased universe (the names that survived to 2026 are disproportionately winners).
This notebook rebuilds the study on the **full historical membership**: every name that was *ever*
in the STI or SGMCNL over 2003–2025 (71 tickers, incl. delisted/removed ones), and at each
rebalance ranks **only the names that were actually in the index that month**.

**Inputs**
- `PricePanel` / `Classification2` (from `PullPricePanelRobust` — 71 names, delisted included)
- `membership_pit.csv` (point-in-time members per year-end, from `PullHistoricalMembership`)
- `macro_monthly.csv` (time-varying risk-free `RF_3M_Ann`)
- `panel_stock.csv` (only to backfill `PX_TO_BOOK` for survivors — see caveat)

**Method held identical to the base model**: total-return realised returns, price momentum
signals, NaN-aware composite, time-varying risk-free, 10bp one-way costs, top/bottom 6.

**Caveats (state these in the write-up)**
1. `PX_TO_BOOK` failed to pull via monthly BDH, so the book-to-market factor exists only for the
   30 survivors; the 41 delisted names lean on dividend-yield and PE for value (the NaN-aware
   composite renormalises). A clean `PX_TO_BOOK` re-pull would remove this asymmetry.
2. Membership uses **year-end** snapshots (Bloomberg `INDX_MWEIGHT_HIST`); within-year index
   changes are approximated. Quarterly snapshots would be more precise.
3. History begins 2003-12 (first membership snapshot); the backtest effectively starts 2006.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from collections import Counter
DATA="." 

In [ ]:
# ---------- 1. Build the 71-name panel ----------
pp = pd.read_excel("SGX2_v3.xlsm", sheet_name="PricePanel")
pp["Date"] = pd.to_datetime(pp["Date"], errors="coerce") + pd.offsets.MonthEnd(0)
pp = pp.dropna(subset=["Date","Ticker"]).sort_values(["Ticker","Date"])
for c in ["PX_LAST","TOT_RETURN","MKT_CAP","DIV_YIELD","PX_VOLUME","PE_RATIO","ROE"]:
    pp[c] = pd.to_numeric(pp[c], errors="coerce")
pp["ROE"]      = pp.groupby("Ticker")["ROE"].ffill()       # forward-fill quarterly fundamentals
pp["PE_RATIO"] = pp.groupby("Ticker")["PE_RATIO"].ffill()
pp = pp.drop(columns=["PX_TO_BOOK"])                       # empty in pull

old = pd.read_csv(f"{DATA}/panel_stock.csv", parse_dates=["Date"]); old["Date"]=old["Date"]+pd.offsets.MonthEnd(0)
pp = pp.merge(old[["Date","Ticker","PX_TO_BOOK"]].dropna(), on=["Date","Ticker"], how="left")  # survivors only
pp = pp.merge(pd.read_excel("SGX2_v3.xlsm", sheet_name="Classification2"), on="Ticker", how="left")
pp["Sector"] = pp["Sector"].fillna("Unknown")

s = pp.sort_values(["Ticker","Date"]).reset_index(drop=True)
s["Return_1M"]=s.groupby("Ticker")["TOT_RETURN"].pct_change(1)
s["Ret_PX_1M"]=s.groupby("Ticker")["PX_LAST"].pct_change(1)
s["MOM_12_1"] =s.groupby("Ticker")["PX_LAST"].pct_change(12).shift(1)
s["MOM_6_1"]  =s.groupby("Ticker")["PX_LAST"].pct_change(6).shift(1)
s["MOM_3_1"]  =s.groupby("Ticker")["PX_LAST"].pct_change(3).shift(1)
s["LogMktCap"]=np.log(s["MKT_CAP"].clip(lower=1))
s["BM_Ratio"] =1.0/s["PX_TO_BOOK"].clip(lower=0.1)
s["LogVolume"]=np.log(s["PX_VOLUME"].clip(lower=1))
s["Amihud"]   =s["Ret_PX_1M"].abs()/s["PX_VOLUME"].clip(lower=1e6)

mac=pd.read_csv(f"{DATA}/macro_monthly.csv", parse_dates=["Date"]); mac["Date"]=mac["Date"]+pd.offsets.MonthEnd(0)
s=s.merge(mac[["Date","RF_3M_Ann","Regime_RiskOn","Regime_SGGrowth"]].drop_duplicates("Date"), on="Date", how="left")
RF_M=mac.dropna(subset=["RF_3M_Ann"]).drop_duplicates("Date").set_index("Date")["RF_3M_Ann"]/12.0
print(f"Panel: {s['Ticker'].nunique()} tickers | {s['Date'].min().date()}->{s['Date'].max().date()}")

In [ ]:
# ---------- 2. Factors + NaN-aware composite ----------
def cs_z(x):
    mu,sig=x.mean(),x.std(); return (x-mu)/sig if sig>1e-9 else x*0.0
def wins(x,lo=.01,hi=.99): return x.clip(x.quantile(lo),x.quantile(hi))
FS={"MOM_12_1":("MOM_12_1",1),"MOM_6_1":("MOM_6_1",1),"MOM_3_1":("MOM_3_1",1),"VALUE_BM":("BM_Ratio",1),
    "VALUE_DY":("DIV_YIELD",1),"VALUE_PE_INV":("PE_RATIO",-1),"QUALITY_ROE":("ROE",1),
    "SIZE_SMALL":("LogMktCap",-1),"LIQ_AMIHUD":("Amihud",-1)}
W={"Z_MOM_12_1":.20,"Z_MOM_6_1":.05,"Z_VALUE_BM":.20,"Z_VALUE_DY":.10,"Z_VALUE_PE_INV":.10,
   "Z_QUALITY_ROE":.20,"Z_SIZE_SMALL":.05,"Z_LIQ_AMIHUD":.10}
p=s[s["Date"]>="2004-01-01"].copy()
for fn,(col,d) in FS.items():
    raw=p.groupby("Date")[col].transform(wins); p[f"Z_{fn}"]=d*p.groupby("Date")[raw.name].transform(cs_z)
zc=list(W); wv=np.array([W[c] for c in zc]); Z=p[zc]; mask=Z.notna()
num=(Z.mul(wv,axis=1)).sum(axis=1,min_count=1); den=(mask.mul(wv,axis=1)).sum(axis=1)
p["Composite"]=np.where(mask.sum(axis=1)>=4, num/den, np.nan)
p=p.drop_duplicates(subset=["Date","Ticker"]).reset_index(drop=True)

In [ ]:
# ---------- 3. Point-in-time membership filter ----------
mem=pd.read_csv(f"{DATA}/membership_pit.csv", parse_dates=["AsOfDate"]); mem["AsOfDate"]=mem["AsOfDate"]+pd.offsets.MonthEnd(0)
snaps=sorted(mem["AsOfDate"].unique())
def active_members(d):
    valid=[x for x in snaps if x<=d]
    return set(mem[mem["AsOfDate"]==valid[-1]]["Ticker"]) if valid else None

In [ ]:
# ---------- 4. Backtest engine ----------
N=6; TC=10
def perf(r,lab):
    rfa=RF_M.reindex(r.index).ffill().fillna(0); ex=r-rfa; c=(1+r).cumprod(); ar=(1+r).prod()**(12/len(r))-1
    rm=c.cummax(); mdd=((c-rm)/rm).min(); dn=ex[ex<0].std()
    return {"Run":lab,"Ann_Return_%":round(ar*100,2),"Ann_Vol_%":round(r.std()*np.sqrt(12)*100,2),
            "Sharpe":round(ex.mean()/ex.std()*np.sqrt(12),3) if ex.std()>0 else 0,
            "Sortino":round(ex.mean()/dn*np.sqrt(12),3) if dn>0 else 0,
            "Max_DD_%":round(mdd*100,2),"Calmar":round(ar/abs(mdd),3) if mdd<0 else 0,"N":len(r)}
def run_bt(panel, pit=False, restrict=None, start="2006-01-01", end="2026-01-31"):
    sub=panel[(panel["Date"]>=start)&(panel["Date"]<=end)].copy()
    dts=sorted(sub["Date"].unique()); rows=[]; pl=[]; ps=[]; held=[]
    for i,d in enumerate(dts[:-1]):
        curr=sub[sub["Date"]==d].dropna(subset=["Composite"])
        if restrict is not None: curr=curr[curr["Ticker"].isin(restrict)]
        if pit:
            am=active_members(d)
            if am is not None: curr=curr[curr["Ticker"].isin(am)]
        curr=curr.sort_values("Composite",ascending=False)
        if len(curr)<10: continue
        lt=curr.head(N)["Ticker"].tolist(); st=curr.tail(N)["Ticker"].tolist()
        nd=dts[i+1]; nr=sub[sub["Date"]==nd].drop_duplicates("Ticker").set_index("Ticker")["Return_1M"]
        lg=nr.reindex(lt).mean(); sg=nr.reindex(st).mean()
        lto=len(set(lt)-set(pl))/N if pl else 1.0; sto=len(set(st)-set(ps))/N if ps else 1.0
        ln=lg-lto*TC/1e4; sn=(-sg)-sto*TC/1e4
        rows.append({"Date":nd,"Long":ln,"LS":ln+sn}); held.append((d,lt)); pl=lt; ps=st
    return pd.DataFrame(rows).set_index("Date"), held

In [ ]:
# ---------- 5. Survivorship comparison ----------
surv=set(old["Ticker"].unique())
btA,_     = run_bt(p, restrict=surv)     # survivor-biased (current 30, no PIT)
btB,heldB = run_bt(p, pit=True)          # survivorship-free (71, PIT-filtered)
res=pd.DataFrame([
    perf(btA["Long"].dropna(),"A: Survivor-only  [Long]"),
    perf(btB["Long"].dropna(),"B: Survivorship-free (PIT) [Long]"),
    perf(btA["LS"].dropna(),  "A: Survivor-only  [L/S]"),
    perf(btB["LS"].dropna(),  "B: Survivorship-free (PIT) [L/S]")])
print(res.to_string(index=False))
print("\nDelisted names the PIT long book actually held:")
cnt=Counter([t for _,lt in heldB for t in lt if t not in surv])
for t,n in cnt.most_common(10): print(f"  {t:18s} {n:3d} months")